<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/iLogos/logo_novafct.png" width="200">

# Departamento de Engenharia Mecânica e Industrial
## Mecânica dos Sólidos II

## Tensões tangenciais em vigas com carregamentos transversais

### Problema 2

Uma viga de madeira com secção transversal de 120 $\times$ 160 mm $^2$ foi reforçada com duas barras de aço de secção 100 $\times$ 10 mm $^2$ como indicado na figura. O módulo de elasticidade da madeira é de 10 GPa e o do aço é 210 GPa. Considerando que cada barra de aço é fixa na viga de madeira através de parafusos espaçados de 400 mm, e que estas ligações suportam forças de corte de 900 N, determine o esforço transverso vertical atuante na ligação.

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au05/P2/MSII_Au05_P2.png"
style="max-height: 100%; max-width: 100%;"/>

### Resolução

In [1]:
import numpy as np
import sympy as sy
from sympy.solvers import solve
import matplotlib.pyplot as plt
import os

cor = '2'
if cor == '1':
    plt.rcParams['axes.facecolor'] = (.15, .15, .15)
    plt.rcParams['figure.facecolor'] = (.15, .15, .15)
    plt.rcParams['font.family'] = 'monospace'
    plt.rcParams['font.size'] = 18
    # plt.rcParams['text.usetex'] = True
    params = {"ytick.color" : (.8, .8, .8),
              "xtick.color" : (.8, .8, .8),
              "grid.color" : (.2, .2, .2),
              "text.color" : (.7, .7, .7),
              "axes.labelcolor" : (.8, .8, .8),
              "axes.edgecolor" : (.15, .15, .15)}
else:
    plt.rcParams['axes.facecolor'] = (.7, .7, .7)
    plt.rcParams['figure.facecolor'] = (.7, .7, .7)
    plt.rcParams['font.family'] = 'monospace'
    plt.rcParams['font.size'] = 18
    # plt.rcParams['text.usetex'] = True
    params = {"ytick.color" : (.1, .1, .1),
              "xtick.color" : (.1, .1, .1),
              "grid.color" : (.2, .2, .2),
              "text.color" : (.1, .1, .1),
              "axes.labelcolor" : (.1, .1, .1),
              "axes.edgecolor" : (.15, .15, .15)}
plt.rcParams.update(params)

# data structure, units: N, mm, MPa
# Create an empty class
class varin: pass

w = varin()
s = varin()
d = varin()

w.b = 120.
w.h = 160.
w.E = 10.e3 # unit: MPa

s.b = 100.
s.h = 10.
s.E = 210.e3 # unit: MPa

d.ht = w.h + 2*s.h
d.paraf = 400. # mm
d.Fco = 900. # N

#### Propriedades da secção transversal

- Razão de módulos considerando a secção transformada em aço:

\begin{equation}
n = \frac{E_\mathrm{wood}}{E_\mathrm{streel}}
\end{equation}

In [2]:
n = w.E/s.E
print(f'n = Ew/Es = {n:.5f}')

n = Ew/Es = 0.04762


- Mantendo a altura da viga (viga composta em altura), qual será a área da região transformada de madeira em aço?

In [3]:
b_w2s = w.b*n
print(f'b_w2s = {b_w2s:.3f} [mm]')
A_w2s = b_w2s*s.h
print(f'A_w2s = {A_w2s:.3f} [mm²]')

b_w2s = 5.714 [mm]
A_w2s = 57.143 [mm²]


- Para a secção homogénea equivalente feita apenas de aço, onde está o centroóide? E qual é o momento de inércia da secção em relação ao eixo $z$?

Pela geometria da secção, o centróide é facilmente determinado no centro geométrico do perfil.
O momento de inércia pode ser obtido por operações booleanas e recorrendo, eventualmente, ao teorema dos eixos paralelos.

In [4]:
def irec(i,j): return i*j**3/12

print('sub-area 1:')
b1, h1 = b_w2s, w.h
Iz1 = irec(b1, h1)
print(f'Iz (sub-area 1) :: (b,h) = ({b1:.2f},{h1:.0f}) :: {Iz1:.3e} [mm⁴]')

print('sub-area 2:')
b2, h2 = s.b, s.h
Izc2 = irec(b2, h2)
print(f'Iz (sub-area 2) :: (b,h) = ({b2:.0f},{h2:.0f}) ::  {Izc2:.3e} [mm⁴]')
A2 = s.b*s.h
print(f'A2 = {A2:.3e} [mm²]')
d2 = w.h/2 + s.h/2
print(f'd2 = {d2:.3e} [mm]')
Iz2 = Izc2 + A2*d2**2
print(f'Iz2 = {Iz2:.3e} [mm⁴]')

print('smaller rectangular area:')
Iz  = Iz1 + 2*Iz2
print(f'Iz (homog. section) = {Iz:.3e} [mm⁴]')

sub-area 1:
Iz (sub-area 1) :: (b,h) = (5.71,160) :: 1.950e+06 [mm⁴]
sub-area 2:
Iz (sub-area 2) :: (b,h) = (100,10) ::  8.333e+03 [mm⁴]
A2 = 1.000e+03 [mm²]
d2 = 8.500e+01 [mm]
Iz2 = 7.233e+06 [mm⁴]
smaller rectangular area:
Iz (homog. section) = 1.642e+07 [mm⁴]


- Esforço transverso ($V$) vertical na ligação:

\begin{equation}
\tau = \frac{V Q}{I_z b}
\quad\wedge\quad
\tau = \frac{F}{A}
\quad\Rightarrow\quad
\frac{F}{A} = \frac{V Q}{I_z b}
\quad\Rightarrow\quad
V = \frac{F I_z b}{Q A}
\end{equation}

onde $A$ é a área resistiva do aperto dos parafusos.

- Momento de área de primeira ordem da área da barra de aço:

\begin{equation}
Q = A_1 \overline{y}_1
\end{equation}

In [5]:
y_1 = w.h/2 + s.h/2
print(f'y = {y_1:.3f} [m]')
A_1 = s.b*s.h
Q = A_1*y_1
print(f'Q = {Q:.3e} [m³]')

v = sy.symbols('v')

A = s.b*d.paraf
V = solve(v - d.Fco*Iz*s.b/Q/A,v)[0]
print(f'V = {V:.1f} [N]')

y = 85.000 [m]
Q = 8.500e+04 [m³]
V = 434.6 [N]


- Esforço transverso ($V$) vertical na ligação:

\begin{equation}
q = \tau b = \frac{V Q}{I_z}
\quad\wedge\quad
\tau = \frac{F}{h}
\quad\Rightarrow\quad
\frac{F}{h} = \frac{V Q}{I_z}
\quad\Rightarrow\quad
V = \frac{I_z }{Q} \left(\frac{I_z }{Q }\right)
\end{equation}

In [11]:
mkV2 = solve(v - (d.Fco/d.paraf)*(Iz/Q),v)[0]
print(f'V = {V2:.1f} [N]')

V = 434.6 [N]


---

Copyright (c) DEMI - FCT NOVA

Interactive computing by <a href="https://jupyter.org/" target="_blank"> <span
style="color:#333399"> Jupyter Notebook </span> </a> &nbsp;|&nbsp;Coded by <a href = "mailto: jmc.xavier@fct.unl.pt">José Xavier</a>

Licensed under  <a href="http://creativecommons.org/licenses/by-sa/4.0/"
target="_blank"> <span style="color:#333399;font-size: 20px"> CC BY-SA 4.0  </span></a>